# Error analysis and efficiency measurement

Produces the raw output for Section 7.7 (error analysis) and Section 9
(inference speed and memory) of the paper.

`notebooks/10_analysis.py --errors` trains one model for 35 epochs (~15 min on
a T4), saves it to `artifacts/analysis_model.pt`, and writes the analysis.
`--efficiency` reuses that checkpoint, so it finishes in seconds.

Outputs: `error_analysis.txt`, `efficiency.txt`, `error_analysis_raw.json`.

## How to run

1. Settings: Accelerator = GPU T4 x2, Internet = On
2. Leave `SMOKE = True` and Run All to check the setup (~2 min)
3. Set `SMOKE = False`
4. Save Version -> Save & Run All (Commit)
5. Collect the output files from the version's Output tab

## Settings

In [ ]:
SMOKE  = True            # True = 2 epochs, just to check the setup. False = the real run.
CPU_TOO = True           # also measure efficiency on CPU (see the CPU cell)

REPO   = 'https://github.com/hatheem-r/project_DNN.git'
BRANCH = 'main'

print('smoke', SMOKE, '| cpu run', CPU_TOO)

## GPU

In [ ]:
import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
assert torch.cuda.is_available(), 'Settings -> Accelerator -> GPU T4 x2'

## Code

In [ ]:
import os, shutil
os.chdir('/kaggle/working')
if os.path.exists('project'):
    shutil.rmtree('project')
!git clone -q -b $BRANCH $REPO project
os.chdir('/kaggle/working/project')
!pip install -q 'datasets<3.0.0' pytorch-crf sentencepiece 2>&1 | tail -1
print('cwd', os.getcwd())

## fastText vectors

About 460 MB, downloaded to `/kaggle/temp` so it is not saved as notebook
output. The script reads the path from `SOLD_VECTORS`.

In [ ]:
import os
os.makedirs('/kaggle/temp/embeddings', exist_ok=True)
VEC = '/kaggle/temp/embeddings/cc.si.300.vec.gz'
if not os.path.exists(VEC):
    !wget -q -O $VEC https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.si.300.vec.gz
os.environ['SOLD_VECTORS'] = VEC
!ls -lh $VEC

## Runner

In [ ]:
import os, subprocess

os.makedirs('results', exist_ok=True)
os.makedirs('artifacts', exist_ok=True)

def run(extra, logfile, env=None):
    args = ['python', 'notebooks/10_analysis.py'] + extra
    print(' '.join(args), flush=True)
    with open(logfile, 'w') as log:
        p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1,
                             env=None if env is None else dict(os.environ, **env))
        for line in p.stdout:
            print(line, end='', flush=True)
            log.write(line); log.flush()
        rc = p.wait()
    print('exit code', rc, flush=True)
    return rc

## 1. Error analysis

Trains the model (~15 min) and writes `results/error_analysis.txt` plus
`results/error_analysis_raw.json`.

In [ ]:
extra = ['--errors'] + (['--epochs', '2'] if SMOKE else [])
run(extra, 'results/error_analysis.txt')

## 2. Efficiency on GPU

Reuses the checkpoint just saved. Measured at batch size 32 on the T4.

In [ ]:
run(['--efficiency'], 'results/efficiency.txt')

## 3. Efficiency on CPU

The script measures on whatever device is visible. Hiding the GPU and pinning
one thread gives a single-thread CPU figure alongside the GPU one.

Still batch size 32, not batch 1 — say which when reporting it.

In [ ]:
if CPU_TOO:
    run(['--efficiency'], 'results/efficiency_cpu.txt',
        env={'CUDA_VISIBLE_DEVICES': '', 'OMP_NUM_THREADS': '1',
             'MKL_NUM_THREADS': '1'})
else:
    print('skipped')

## Save the output

In [ ]:
import os, shutil

CKPT = 'artifacts/analysis_model.pt'

if SMOKE:
    # the smoke checkpoint is undertrained, and get_model() would reuse it
    if os.path.exists(CKPT):
        os.remove(CKPT)
    print('Setup OK. Set SMOKE = False, then Save Version -> Save & Run All.')
else:
    for f in ('results/error_analysis.txt', 'results/efficiency.txt',
              'results/efficiency_cpu.txt', 'results/error_analysis_raw.json'):
        if os.path.exists(f):
            shutil.copy(f, '/kaggle/working/')
            print('saved', os.path.basename(f), os.path.getsize(f), 'bytes')

## Report

**Section 2 of the error analysis, ERRORS BY WORD FAMILIARITY.** Compare
unseen-word F1 against seen-word F1 and report the gap either way.

**Section 4, the example tweets.** `[word]` is a hit, `<word>` a false alarm,
`{word}` a miss. Pick five to eight and write one or two sentences each on why
the model behaved that way. This needs someone who reads Sinhala.

**Section 9 numbers.** Report as measured, and state hardware and batch size.

Then put the output files in `results/` and commit:

```
git add results/error_analysis.txt results/efficiency.txt results/error_analysis_raw.json
git commit -m "error analysis and efficiency measurements"
git pull --rebase
git push
```